<div style="background-color:#118AB2; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>

# <h2 style="border-bottom: 4px solid #118AB2; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">Model Selection</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)


## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1. Imports and data loading</h3>

This notebook makes use of the previously transformed and processed datasets to train and evaluate different prediction models.


In [ ]:
"""Importing the libraries needed for data handling, preprocessing, modeling, and evaluation"""
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from datetime import datetime

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    StackingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from IPython.display import display
import optuna
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import cross_val_score, StratifiedKFold

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


'''Importing modeling utility functions'''
from utils.utils_modeling import optimize_with_optuna, train_all_models


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress the output for each trial optuna
#warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")  # Run to remove warnings
#warnings.resetwarnings()  # Run to add warnings


In [58]:
"""Loading the features datasets"""
train_data = pd.read_csv(PROJECT_ROOT / "project_data" / "X_train_clean.csv", index_col=0)
val_data = pd.read_csv(PROJECT_ROOT / "project_data" / "X_val_clean.csv", index_col=0)
test_data = pd.read_csv(PROJECT_ROOT / "project_data" / "X_test_clean.csv", index_col=0)

'''Loading the y datasets'''
y_train = pd.read_csv(PROJECT_ROOT / "project_data" / "y_train_clean.csv")
y_val = pd.read_csv(PROJECT_ROOT / "project_data" / "y_val_clean.csv")
y_test = pd.read_csv(PROJECT_ROOT / "project_data" / "donors_test.csv")



## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2. Defining Models</h3>

Defining standalone models to be tested

In [ ]:
def optimize_gradient_boosting(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("GradientBoosting", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_logistic_regression(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("LogisticRegression", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_adaboost(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("AdaBoost", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_decision_tree(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("DecisionTree", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_random_forest(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("RandomForestClassifier", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_gaussian_nb(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("GaussianNB", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_knn(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("KNeighborsClassifier", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_mlp(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("MLPClassifier", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

def optimize_stacking(train_data, y_train, val_data, y_val, n_trials=100, **kwargs):
    return optimize_with_optuna("Stacking", train_data, y_train, val_data, y_val, n_trials=n_trials, **kwargs)

# Usage
study = optimize_logistic_regression(
    train_data, y_train, val_data, y_val,
    n_trials=150, hyper_importance=True, slice_plot=True, optimization_history=True
)

model_optimizers = {
    "GradientBoosting": optimize_gradient_boosting,
    "LogisticRegression": optimize_logistic_regression,
    "AdaBoost": optimize_adaboost,
    "DecisionTree": optimize_decision_tree,
    "RandomForestClassifier": optimize_random_forest,
    "GaussianNB": optimize_gaussian_nb,
    "KNeighborsClassifier": optimize_knn,
    "MLPClassifier": optimize_mlp,
    "Stacking": optimize_stacking,
}

print("Defined model optimization wrappers for:", ", ".join(model_optimizers.keys()))

Defined model optimization wrappers for: GradientBoosting, LogisticRegression, AdaBoost, DecisionTree, RandomForestClassifier, GaussianNB, KNeighborsClassifier, MLPClassifier, Stacking


## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">4. Training and Evaluating Models</h3>

Training and tuning the previously defined models

In [ ]:
results_df = train_all_models(
    train_data, y_train, val_data, y_val, test_data,
    n_trials=100
)
results_df

In [ ]:

#Comparing results
print("=== Model Comparison ===")
display(results_df)

print("\nBest Model:")
best = results_df.iloc[0]
print(f"  Model:     {best['MODEL']}")
print(f"  F1 Score:  {best['F1 SCORE']}")
print(f"  Accuracy:  {best['ACCURACY']}")
print(f"  Precision: {best['PRECISION']}")
print(f"  Recall:    {best['RECALL']}")
print(f"  Params:    {best['PARAMETERS']}")

## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">5. Model Selection</h3>



### Chosen Model
> Notes: why that model was chosen

In [ ]:
best_model_name   = results_df.iloc[0]['MODEL']
best_model_params = results_df.iloc[0]['PARAMETERS']
best_f1           = results_df.iloc[0]['F1 SCORE']

entry = f"""
---
**Date:** {datetime.now().strftime("%Y-%m-%d %H:%M")}  
**Model:** {best_model_name}  
**F1 Score (val):** {best_f1}  
**Parameters:** {best_model_params}  
"""

with open("results_history.md", "a") as f:
    f.write(entry)

Writting this runs results to results_history.md, for record keeping

In [ ]:
entry = f"""
---
**Date:** {datetime.now().strftime("%Y-%m-%d %H:%M")}  
**Model:** {best_model_name}  
**F1 Score (val):** {best_f1}  
**Parameters:** {best_model_params}  
"""

with open(PROJECT_ROOT /"results_history.md", "a") as f:
    f.write(entry)